<a href="https://colab.research.google.com/github/Fernando0828/INTELIGENCIA-ARTIFICIAL-2/blob/main/prediccion_valor_casas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción del valor de casas — Regresión Lineal vs Árbol de Decisión

**Dataset:** [Housing Prices Dataset (Kaggle)](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset)

**Objetivo:** construir un modelo que prediga el precio (`price`) de una vivienda a partir de sus
características (área, número de habitaciones, baños, si tiene aire acondicionado, etc.), y comparar
el rendimiento de dos algoritmos distintos:

1. **Regresión Lineal** — un modelo simple que asume una relación lineal entre las variables y el precio.
2. **Árbol de Decisión (Decision Tree Regressor)** — un modelo que puede capturar relaciones no lineales
   dividiendo los datos en reglas de decisión.

A lo largo del notebook voy dejando comentarios explicando **por qué** hago cada paso, no solo qué hace
la línea de código, tal como lo pide el ejercicio de referencia de Kaggle
([exercise-introduction](https://www.kaggle.com/code/alexisbcook/exercise-introduction)).


## 1. Importar librerías

Todo lo que voy a necesitar durante el notebook: manejo de datos, gráficos, y las herramientas de `scikit-learn` para modelar y evaluar.

In [1]:
# pandas: la uso para cargar el csv en un DataFrame y para todas las operaciones de limpieza/exploración
import pandas as pd

# numpy: lo uso para operaciones numéricas (por ejemplo raíz cuadrada al calcular el RMSE)
import numpy as np

# matplotlib y seaborn: los uso para graficar (histogramas, mapa de calor de correlación, comparación de modelos)
import matplotlib.pyplot as plt
import seaborn as sns

# train_test_split: lo necesito para separar el dataset en un set de entrenamiento y uno de prueba,
# así puedo evaluar el modelo con datos que NO vio durante el entrenamiento
from sklearn.model_selection import train_test_split

# LinearRegression: el primer modelo que voy a comparar
from sklearn.linear_model import LinearRegression

# DecisionTreeRegressor: el segundo modelo que voy a comparar
from sklearn.tree import DecisionTreeRegressor, plot_tree

# métricas de regresión: las uso para medir qué tan bien predice cada modelo
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# fijo una semilla para que los resultados sean reproducibles (si vuelvo a correr el notebook, obtengo lo mismo)
SEED = 42

# configuro el estilo de los gráficos para que se vean más limpios
sns.set_style("whitegrid")


## 2. Cargar el dataset

El dataset original está en Kaggle: **Housing Prices Dataset** (archivo `Housing.csv`, 545 filas, 13 columnas).

Como Kaggle pide autenticación para descargar por API, en Colab lo más simple es subir el archivo manualmente:

1. Entro a https://www.kaggle.com/datasets/yasserh/housing-prices-dataset y descargo `Housing.csv`.
2. Corro la celda de abajo y selecciono el archivo cuando me lo pida el botón de carga.

*(Si prefiero no subirlo a mano cada vez, dejo comentada una alternativa con `kagglehub`, que descarga el
dataset directamente si tengo configurado mi `kaggle.json` en Colab).*


In [ ]:
# --- Opción A (recomendada): subir el archivo manualmente ---
# files.upload() abre un botón en Colab para seleccionar el Housing.csv desde mi computador
from google.colab import files
uploaded = files.upload()  # esto me devuelve un diccionario con el nombre del archivo subido

# tomo el primer (y único) archivo que subí y lo guardo en una variable
filename = list(uploaded.keys())[0]

# --- Opción B (alternativa, descomentar si tengo kaggle.json configurado): ---
# import kagglehub
# path = kagglehub.dataset_download("yasserh/housing-prices-dataset")
# filename = f"{path}/Housing.csv"

# leo el csv y lo cargo en un DataFrame de pandas, que es la estructura que voy a usar en todo el notebook
df = pd.read_csv(filename)

# reviso las primeras 5 filas para confirmar que se cargó bien y ver cómo lucen los datos
df.head()


## 3. Exploración de datos (EDA)

Antes de modelar, necesito entender qué tipo de datos tengo, si hay valores nulos, y cómo se relacionan las variables con el precio.

In [ ]:
# .shape me dice cuántas filas y columnas tiene el dataset -> (filas, columnas)
print("Dimensiones del dataset:", df.shape)

# .info() me muestra el tipo de dato de cada columna (numérico u objeto/texto) y si hay valores nulos
df.info()


In [ ]:
# isnull().sum() cuenta cuántos valores nulos hay por columna
# esto es clave: si hubiera nulos tendría que decidir si los relleno o elimino antes de entrenar el modelo
df.isnull().sum()


In [ ]:
# describe() me da estadísticas básicas (media, mínimo, máximo, etc.) de las columnas numéricas
# esto me ayuda a detectar valores extremos (outliers) o rangos raros en el precio o el área
df.describe()


In [ ]:
# separo las columnas según su tipo, porque las voy a tratar distinto más adelante:
# - columnas categóricas (texto: 'yes'/'no', tipo de amoblado, etc.)
# - columnas numéricas (área, habitaciones, baños, pisos, parqueaderos, precio)
cat_cols = df.select_dtypes(include="object").columns.tolist()
num_cols = df.select_dtypes(exclude="object").columns.tolist()

print("Columnas categóricas:", cat_cols)
print("Columnas numéricas:", num_cols)


In [ ]:
# histograma del precio: quiero ver cómo se distribuye la variable que voy a predecir
plt.figure(figsize=(8, 4))
sns.histplot(df["price"], kde=True)  # kde=True agrega una curva de densidad suavizada encima del histograma
plt.title("Distribución del precio de las viviendas")
plt.xlabel("Precio")
plt.show()


In [ ]:
# mapa de calor de correlación entre las variables numéricas
# esto me da una primera pista de qué variables se relacionan más fuerte con el precio
plt.figure(figsize=(8, 6))
corr = df[num_cols].corr()  # calculo la matriz de correlación solo con las columnas numéricas
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")  # annot=True muestra el número dentro de cada celda
plt.title("Correlación entre variables numéricas")
plt.show()


**Análisis rápido de la EDA:** el dataset no tiene valores nulos, así que no necesito imputar ni
eliminar filas por esa razón. El precio se ve razonablemente distribuido (con algo de sesgo hacia la
derecha, es decir, algunas casas bastante más caras que el resto). En la matriz de correlación, `area` es
la variable numérica que más se relaciona con `price`, lo cual tiene sentido: entre más grande la casa,
más cara suele ser.

## 4. Preprocesamiento

Los modelos de `scikit-learn` solo entienden números, así que tengo que convertir las columnas de texto
(`yes`/`no`, tipo de amoblado) a valores numéricos antes de entrenar.

In [ ]:
# hago una copia del dataframe original para no modificar los datos crudos por accidente
df_model = df.copy()

# columnas binarias tipo 'yes'/'no': las mapeo a 1/0 manualmente porque son solo dos categorías
binary_cols = [c for c in cat_cols if set(df_model[c].unique()) <= {"yes", "no"}]
for c in binary_cols:
    df_model[c] = df_model[c].map({"yes": 1, "no": 0})  # reemplazo 'yes' por 1 y 'no' por 0

print("Columnas binarias convertidas:", binary_cols)
df_model[binary_cols].head()


In [ ]:
# columnas categóricas con más de dos valores (por ejemplo 'furnishingstatus'):
# uso One-Hot Encoding (pd.get_dummies) para no inventar un orden que no existe entre las categorías
multi_cat_cols = [c for c in cat_cols if c not in binary_cols]
print("Columnas categóricas con más de 2 valores:", multi_cat_cols)

# drop_first=True evita crear una columna redundante (evita colinealidad perfecta entre las dummies)
df_model = pd.get_dummies(df_model, columns=multi_cat_cols, drop_first=True)

df_model.head()


In [ ]:
# separo las variables predictoras (X) de la variable objetivo (y)
X = df_model.drop(columns=["price"])  # X = todas las columnas menos el precio, que es lo que quiero predecir
y = df_model["price"]                 # y = la columna que quiero predecir

# divido en set de entrenamiento (80%) y de prueba (20%)
# el modelo aprende SOLO con el set de entrenamiento, y lo evalúo con el set de prueba que nunca vio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print("Filas de entrenamiento:", X_train.shape[0])
print("Filas de prueba:", X_test.shape[0])


## 5. Modelo 1 — Regresión Lineal

Empiezo con el modelo más simple: la regresión lineal ajusta una línea (en realidad un hiperplano, porque
hay varias variables) que minimiza el error entre el precio real y el precio predicho.

In [ ]:
# creo una instancia del modelo de regresión lineal
lin_reg = LinearRegression()

# entreno el modelo: aquí es donde 'aprende' los coeficientes que mejor ajustan X_train a y_train
lin_reg.fit(X_train, y_train)

# genero las predicciones del modelo sobre el set de prueba (datos que el modelo no vio al entrenar)
y_pred_lin = lin_reg.predict(X_test)


In [ ]:
# MAE (Error Absoluto Medio): el promedio de qué tan lejos está la predicción del valor real, en la misma unidad que el precio
mae_lin = mean_absolute_error(y_test, y_pred_lin)

# MSE (Error Cuadrático Medio): igual que el MAE pero penaliza más fuerte los errores grandes
mse_lin = mean_squared_error(y_test, y_pred_lin)

# RMSE: raíz cuadrada del MSE, para volver a tener la métrica en la misma unidad que el precio (más interpretable que el MSE)
rmse_lin = np.sqrt(mse_lin)

# R^2: qué proporción de la variación del precio logra explicar el modelo (1.0 sería predicción perfecta)
r2_lin = r2_score(y_test, y_pred_lin)

print(f"Regresión Lineal -> MAE: {mae_lin:,.0f} | RMSE: {rmse_lin:,.0f} | R2: {r2_lin:.3f}")


In [ ]:
# reviso los coeficientes del modelo: me dicen cuánto cambia el precio por cada unidad que cambia cada variable
# (manteniendo las demás variables constantes). Esto me ayuda a interpretar el modelo, no solo evaluarlo.
coef_df = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": lin_reg.coef_
}).sort_values("coeficiente", key=abs, ascending=False)  # ordeno por magnitud del coeficiente, sin importar el signo

coef_df


**Análisis:** el coeficiente más grande suele ser el de `area`, lo que confirma lo que vi en la
matriz de correlación: el tamaño de la casa es el factor que más mueve el precio en un modelo lineal.
Los coeficientes positivos (aire acondicionado, área preferencial, etc.) suben el precio esperado; si
alguno fuera negativo, indicaría que esa característica está asociada a un precio más bajo, manteniendo
todo lo demás igual.

## 6. Modelo 2 — Árbol de Decisión

Ahora entreno un árbol de decisión, que en lugar de ajustar una línea, va dividiendo los datos en
preguntas tipo "¿el área es mayor a X?" hasta llegar a un valor de predicción. Puede capturar relaciones
no lineales que la regresión lineal no puede.

In [ ]:
# max_depth=5: limito la profundidad del árbol a propósito.
# Un árbol sin límite de profundidad memoriza el set de entrenamiento (overfitting) y generaliza mal.
tree_reg = DecisionTreeRegressor(max_depth=5, random_state=SEED)

# entreno el árbol con los mismos datos de entrenamiento que usé en la regresión lineal, para que la comparación sea justa
tree_reg.fit(X_train, y_train)

# genero las predicciones sobre el mismo set de prueba
y_pred_tree = tree_reg.predict(X_test)


In [ ]:
# calculo las mismas métricas que usé para la regresión lineal, así puedo comparar manzanas con manzanas
mae_tree = mean_absolute_error(y_test, y_pred_tree)
mse_tree = mean_squared_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mse_tree)
r2_tree = r2_score(y_test, y_pred_tree)

print(f"Árbol de Decisión -> MAE: {mae_tree:,.0f} | RMSE: {rmse_tree:,.0f} | R2: {r2_tree:.3f}")


In [ ]:
# feature_importances_ me dice qué variables usó más el árbol para dividir los datos
# (a diferencia de los coeficientes lineales, esto no tiene signo, solo indica "cuánta importancia" tuvo cada variable)
importance_df = pd.DataFrame({
    "variable": X.columns,
    "importancia": tree_reg.feature_importances_
}).sort_values("importancia", ascending=False)

importance_df


In [ ]:
# grafico el árbol (limitado a los primeros 3 niveles para que se pueda leer) para entender visualmente cómo decide
plt.figure(figsize=(20, 8))
plot_tree(tree_reg, feature_names=X.columns, max_depth=2, filled=True, fontsize=8)
plt.title("Árbol de decisión (primeros niveles)")
plt.show()


**Análisis:** normalmente `area` también aparece como la variable más importante para el árbol,
coincidiendo con la regresión lineal. La diferencia es que el árbol puede combinar variables de forma no
lineal (por ejemplo, "si el área es grande Y tiene aire acondicionado, el precio sube mucho más"), algo
que la regresión lineal no puede representar directamente porque asume que el efecto de cada variable es
siempre el mismo, sin importar el valor de las demás.

## 7. Comparación de los dos modelos

Junto las métricas de ambos modelos en una sola tabla y en un gráfico para compararlos lado a lado.

In [ ]:
# armo un DataFrame con las métricas de ambos modelos para comparar fácilmente
resultados = pd.DataFrame({
    "Modelo": ["Regresión Lineal", "Árbol de Decisión"],
    "MAE": [mae_lin, mae_tree],
    "RMSE": [rmse_lin, rmse_tree],
    "R2": [r2_lin, r2_tree]
})

resultados


In [ ]:
# grafico de barras comparando el RMSE y el R2 de ambos modelos, que son las métricas más fáciles de interpretar
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# RMSE más bajo = mejor (menos error en la predicción)
sns.barplot(data=resultados, x="Modelo", y="RMSE", ax=axes[0])
axes[0].set_title("RMSE por modelo (más bajo = mejor)")

# R2 más alto (cerca de 1) = mejor (explica mejor la variación del precio)
sns.barplot(data=resultados, x="Modelo", y="R2", ax=axes[1])
axes[1].set_title("R² por modelo (más alto = mejor)")

plt.tight_layout()
plt.show()


## 8. Conclusiones

- **Regresión Lineal:** funciona bien como punto de partida porque es simple e interpretable (puedo ver
  exactamente cuánto aporta cada variable con sus coeficientes), pero asume que la relación entre las
  variables y el precio es siempre lineal, lo cual es una simplificación fuerte para datos de vivienda.
- **Árbol de Decisión:** puede capturar relaciones no lineales y combinaciones entre variables, pero es
  más sensible a *overfitting* si lo dejo crecer sin límite de profundidad (por eso limité `max_depth`).
- Según las métricas obtenidas en `resultados` (`MAE`, `RMSE`, `R2`), el modelo con **menor RMSE y mayor
  R²** es el que generaliza mejor sobre datos que no vio durante el entrenamiento. *(Reviso los números
  que arrojó mi ejecución para completar aquí cuál de los dos ganó y, con mis propias palabras, por qué
  creo que pasó eso con este dataset en particular.)*
- Como siguiente paso, podría probar afinar los hiperparámetros del árbol (`max_depth`, `min_samples_leaf`)
  con `GridSearchCV`, o probar un modelo de ensamble como `RandomForestRegressor`, que suele mejorar sobre
  un único árbol de decisión.
